# Week 2: Mini Semantic Search Lab

This notebook implements a complete semantic search pipeline for FAQs.

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.cleaning import load_raw_data, handle_missing_values, remove_duplicates, normalize_text_case_whitespace, convert_to_appropriate_dtypes, detect_handle_outliers
from src.numpy_stats import compute_text_length_stats
from src.semantic_search import MiniSemanticSearch


## 2. Load Raw Dataset

In [ ]:
raw_df = load_raw_data('../data/raw_faq.csv')
print('Raw dataset shape:', raw_df.shape)
raw_df


## 3. Data Cleaning (5 Functions)

In [ ]:
print('=== Before Cleaning ===')
print('Shape:', raw_df.shape)
print(raw_df)

df1 = handle_missing_values(raw_df)
print('\n=== After handling missing values ===')
print('Shape:', df1.shape)

df2 = remove_duplicates(df1)
print('\n=== After removing duplicates ===')
print('Shape:', df2.shape)

df3 = normalize_text_case_whitespace(df2)
print('\n=== After normalizing text case and whitespace ===')
print(df3[['question', 'answer']])

df4 = convert_to_appropriate_dtypes(df3)
print('\n=== After converting dtypes ===')
print(df4.dtypes)

df_clean = detect_handle_outliers(df4)
print('\n=== After outlier detection/handling ===')
print('Shape:', df_clean.shape)
print(df_clean[['question', 'answer']])


## 4. NumPy Calculations (Statistical Analysis)

In [ ]:
q_stats, q_lengths = compute_text_length_stats(df_clean, text_col='question')
print('=== Question Text Length Statistics (NumPy) ===')
for k, v in q_stats.items():
    print(k + ': ' + str(v))

a_stats, a_lengths = compute_text_length_stats(df_clean, text_col='answer')
print('\n=== Answer Text Length Statistics (NumPy) ===')
for k, v in a_stats.items():
    print(k + ': ' + str(v))

print('\n=== Additional NumPy Calculations ===')
print('Mean of question lengths:', np.mean(q_lengths))
print('Median of question lengths:', np.median(q_lengths))
print('Standard deviation:', np.std(q_lengths, ddof=1))
print('90th percentile:', np.percentile(q_lengths, 90))
print('Variance:', np.var(q_lengths, ddof=1))
print('Sum of all question chars:', np.sum(q_lengths))


## 5. Build Semantic Search with Embeddings and ChromaDB

In [ ]:
searcher = MiniSemanticSearch(persist_directory='../chroma_db')
num_added = searcher.add_documents(df_clean)
print('Added ' + str(num_added) + ' documents to ChromaDB collection')


## 6. Example Semantic Searches (Top 3 Results)

In [ ]:
query1 = 'What is machine learning?'
print('Query: ' + query1)
print('='*60)
results1 = searcher.search(query1, n_results=3)
for i in range(len(results1['metadatas'][0])):
    print('\nResult ' + str(i+1) + ':')
    print('Question: ' + str(results1['metadatas'][0][i]['question']))
    print('Answer: ' + str(results1['metadatas'][0][i]['answer']))
    print('Distance: ' + str(results1['distances'][0][i]))


In [ ]:
query2 = 'How do computers understand language?'
print('Query: ' + query2)
print('='*60)
results2 = searcher.search(query2, n_results=3)
for i in range(len(results2['metadatas'][0])):
    print('\nResult ' + str(i+1) + ':')
    print('Question: ' + str(results2['metadatas'][0][i]['question']))
    print('Answer: ' + str(results2['metadatas'][0][i]['answer']))
    print('Distance: ' + str(results2['distances'][0][i]))


In [ ]:
query3 = 'teaching models from examples'
print('Query: ' + query3)
print('='*60)
results3 = searcher.search(query3, n_results=3)
for i in range(len(results3['metadatas'][0])):
    print('\nResult ' + str(i+1) + ':')
    print('Question: ' + str(results3['metadatas'][0][i]['question']))
    print('Answer: ' + str(results3['metadatas'][0][i]['answer']))
    print('Distance: ' + str(results3['distances'][0][i]))
